In [1]:
import pandas as pd
import numpy as np
import torch
from sklearn import preprocessing
from dgl.data import DGLDataset
import dgl
import time
import networkx as nx
import category_encoders as ce
import torch.nn as nn
import torch.nn.functional as F
import dgl.function as fn
import torch
import tqdm
import math

from typing import *
from sklearn.preprocessing import StandardScaler, Normalizer
import socket
import struct
import random
from sklearn.model_selection import train_test_split
import gc

import torch.optim as optim
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, f1_score
from sklearn.ensemble import IsolationForest
import itertools
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

graphs, _ = dgl.load_graphs('train.bin')
train_g = graphs[0]
train_g = train_g.to(device)

class SAGELayer(nn.Module):
    def __init__(self, ndim_in, edims, ndim_out, activation):
      super(SAGELayer, self).__init__()
      self.W_apply = nn.Linear(ndim_in + ndim_out , ndim_out)
      self.activation = F.relu
      self.W_msg = nn.Linear(ndim_in + edims, ndim_out)
      self.W_edge = nn.Linear(ndim_out * 2 + edims, 64)
      self.reset_parameters()

    def reset_parameters(self):
      gain = nn.init.calculate_gain('relu')
      nn.init.xavier_uniform_(self.W_apply.weight, gain=gain)

    def message_func(self, edges):
      # return {'m':  edges.data['h']}
        return {'m': self.W_msg(torch.cat([edges.data['h'], edges.dst['h']], 2))}

    def forward(self, g_dgl, nfeats, efeats):
      with g_dgl.local_scope():
        g = g_dgl
        g.ndata['h'] = nfeats
        g.edata['h'] = efeats
        g.update_all(self.message_func, fn.mean('m', 'h_neigh'))
        g.ndata['h'] = F.relu(self.W_apply(torch.cat([g.ndata['h'], g.ndata['h_neigh']], 2)))
        u, v = g.edges()
        src_feat = g.srcdata['h'][u]  # shape: [num_edges, feature_dim]
        dst_feat = g.dstdata['h'][v]  # shape: [num_edges, feature_dim]
        edge_feat = efeats      # shape: [num_edges, edge_feature_dim]

        # 拼接：[源节点特征，边特征，目标节点特征]
        edge_fin = torch.cat((src_feat, edge_feat, dst_feat), dim=2)

        # （如果你要喂进一个线性层，比如 self.W_edge）
        edge = self.W_edge(edge_fin)
        # u, v = g.edges()
        # edge = self.W_edge(torch.cat((g.srcdata['h'][u], g.dstdata['h'][v]), 2))
        return g.ndata['h'], edge
  
    
class SAGE(nn.Module):
    def __init__(self, ndim_in, ndim_out, edim,  activation):
      super(SAGE, self).__init__()
      # self.atten = nn.Parameter(torch.randn(1, edim))
      self.layers = nn.ModuleList()
      self.layers.append(SAGELayer(ndim_in, edim, ndim_out, F.relu))

    def forward(self, g, nfeats, efeats, corrupt=False):
      # efeats = efeats * self.atten
      if corrupt:
        e_perm = torch.randperm(g.number_of_edges())
        #n_perm = torch.randperm(g.number_of_nodes())
        efeats = efeats[e_perm]
        #nfeats = nfeats[n_perm]
      for i, layer in enumerate(self.layers):
        #nfeats = layer(g, nfeats, efeats)
        nfeats, e_feats = layer(g, nfeats, efeats)
      #return nfeats.sum(1)
      return nfeats.sum(1), e_feats.sum(1)
    
class DGI(nn.Module):
    def __init__(self, ndim_in, ndim_out, edim, activation):
      super(DGI, self).__init__()
      self.encoder = SAGE(ndim_in, ndim_out, edim,  F.relu)
      # self.discriminator = Discriminator(ndim_out)
      self.loss = nn.BCEWithLogitsLoss()

    def forward(self, g, n_features, e_features):
      positive = self.encoder(g, n_features, e_features, corrupt=False)
      negative = self.encoder(g, n_features, e_features, corrupt=True)

      positive = positive[1]
      negative = negative[1]

#       summary = torch.sigmoid(positive.mean(dim=0))

      # positive = self.discriminator(positive, summary)
      # negative = self.discriminator(negative, summary)

      l1 = self.loss(positive,  torch.full_like(positive, 100))
      l2 = self.loss(negative,  torch.full_like(negative, 1))

      return l1 + l2
    
ndim_in = train_g.ndata['h'].shape[2]
# hidden_features = 128
ndim_out = 128
# num_layers = 1
edim = train_g.edata['h'].shape[2]
learning_rate = 1e-3
# epochs = 4000

dgi = DGI(ndim_in,
    ndim_out,
    edim,
    F.relu).to(device)

dgi_optimizer = torch.optim.Adam(dgi.parameters(),
                lr=1e-3,
                weight_decay=0.)

epochs = 51
cnt_wait = 0
best = 1e9
best_t = 0
dur = []
node_features = train_g.ndata['h']
edge_features = train_g.edata['h']

dgi.train()

for epoch in range(epochs):
    
    
    t0 = time.time()

    dgi_optimizer.zero_grad()
    loss = dgi(train_g, node_features, edge_features)
    loss.backward()
    dgi_optimizer.step()

#     if loss < best:
#         best = loss
#         best_t = epoch
#         cnt_wait = 0
#         torch.save(dgi.state_dict(), 'best_dgi.pkl')
#     else:
#         cnt_wait += 1

#     if cnt_wait == patience:
#       print('Early stopping!')
#       break

   
    dur.append(time.time() - t0)

    if epoch % 50 == 0:

        print("Epoch {:05d} | Time(s) {:.4f} | Loss {:.4f} ".format(epoch, np.mean(dur),
              loss.item()))

Epoch 00000 | Time(s) 0.7217 | Loss 2.4763 
Epoch 00050 | Time(s) 0.3501 | Loss -21893.3789 


In [2]:
file_name = "nf-cic-v2-0.02_004.csv"
data = pd.read_csv(file_name)

lab_enc = preprocessing.LabelEncoder()
lab_enc.fit(data["Attack"])

training_emb = dgi.encoder(train_g, train_g.ndata['h'], train_g.edata['h'])[1]
training_emb = training_emb.detach().cpu().numpy()

graphs, _ = dgl.load_graphs('test.bin')
test_g = graphs[0]
test_g = test_g.to(device)

testing_emb = dgi.encoder(test_g, test_g.ndata['h'], test_g.edata['h'])[1]
testing_emb = testing_emb.detach().cpu().numpy()

df_train = pd.DataFrame(training_emb, )
df_train["Attack"] = lab_enc.inverse_transform(
        train_g.edata['Attack'].detach().cpu().numpy())
df_train["Label"] = train_g.edata['Label'].detach().cpu().numpy()

df_test = pd.DataFrame(testing_emb, )
df_test["Attack"] = lab_enc.inverse_transform(
        test_g.edata['Attack'].detach().cpu().numpy())
df_test["Label"] = test_g.edata['Label'].detach().cpu().numpy()

benign_train_samples = df_train[df_train.Label == 0].drop(columns=["Label", "Attack"])
normal_train_samples = df_train.drop(columns=["Label", "Attack"])

train_labels = df_train["Label"]
test_labels = df_test["Label"]

test_samples = df_test.drop(columns=["Label", "Attack"])

contamination = [ 0.01, 0.04]

from pyod.models.cblof import CBLOF
n_est = [3,5]
params = list(itertools.product(n_est, contamination))
score = -1
bs = None
for n_est, con in tqdm.tqdm(params):
    
    clf_if = CBLOF(n_clusters=n_est, contamination=con)
    clf_if.fit(benign_train_samples)
    y_pred = clf_if.predict(test_samples)
    test_pred = y_pred

    f1 = f1_score(test_labels, test_pred, average='macro')

    if f1 > score:
        score = f1
        best_params = {'n_estimators': n_est,
                       "con": con
                }
        bs = test_pred
    del clf_if
    gc.collect()


print(best_params)
print(score)
print(classification_report(test_labels, bs, digits=4))

100%|██████████| 4/4 [00:26<00:00,  6.63s/it]


{'n_estimators': 5, 'con': 0.01}
0.5007268732688622
              precision    recall  f1-score   support

           0     0.9610    0.9870    0.9738    199328
           1     0.0548    0.0185    0.0276      8130

    accuracy                         0.9491    207458
   macro avg     0.5079    0.5027    0.5007    207458
weighted avg     0.9255    0.9491    0.9368    207458



In [3]:
n_est = [3,5]
contamination = [ 0.01, 0.04]
params = list(itertools.product(n_est, contamination))
score = -1
bs = None
for n_est, con in tqdm.tqdm(params):
    
    clf_if = CBLOF(n_clusters=n_est, contamination=con)
    clf_if.fit(normal_train_samples)
    y_pred = clf_if.predict(test_samples)
    test_pred = y_pred

    f1 = f1_score(test_labels, test_pred, average='macro')

    if f1 > score:
        score = f1
        best_params = {'n_estimators': n_est,
                       "con": con
                }
        bs = test_pred
    del clf_if
    gc.collect()


print(best_params)
print(score)
print(classification_report(test_labels, bs, digits=4))

100%|██████████| 4/4 [00:24<00:00,  6.12s/it]


{'n_estimators': 5, 'con': 0.01}
0.5009487245113725
              precision    recall  f1-score   support

           0     0.9610    0.9875    0.9741    199328
           1     0.0566    0.0185    0.0278      8130

    accuracy                         0.9495    207458
   macro avg     0.5088    0.5030    0.5009    207458
weighted avg     0.9256    0.9495    0.9370    207458



In [4]:
from pyod.models.hbos import HBOS

n_est = [5,10]
params = list(itertools.product(n_est, contamination))
score = -1
bs = None
for n_est, con in tqdm.tqdm(params):
    
    clf_if = HBOS(n_bins=n_est, contamination=con)
    clf_if.fit(benign_train_samples)
    y_pred = clf_if.predict(test_samples)
    test_pred = y_pred

    f1 = f1_score(test_labels, test_pred, average='macro')

    if f1 > score:
        score = f1
        best_params = {'n_estimators': n_est,
                       "con": con
                }
        bs = test_pred
    del clf_if
    gc.collect()


print(best_params)
print(score)
print(classification_report(test_labels, bs, digits=4))

100%|██████████| 4/4 [00:13<00:00,  3.39s/it]


{'n_estimators': 10, 'con': 0.04}
0.585768530110296
              precision    recall  f1-score   support

           0     0.9685    0.9581    0.9632    199328
           1     0.1865    0.2358    0.2083      8130

    accuracy                         0.9298    207458
   macro avg     0.5775    0.5969    0.5858    207458
weighted avg     0.9378    0.9298    0.9337    207458



In [5]:
n_est = [5,10]
# contamination = [0.001, 0.01, 0.04, 0.05, 0.1, 0.2]
params = list(itertools.product(n_est, contamination))
score = -1
bs = None
for n_est, con in tqdm.tqdm(params):
    
    clf_if = HBOS(n_bins=n_est, contamination=con)
    clf_if.fit(normal_train_samples)
    y_pred = clf_if.predict(test_samples)
    test_pred = y_pred

    f1 = f1_score(test_labels, test_pred, average='macro')

    if f1 > score:
        score = f1
        best_params = {'n_estimators': n_est,
                       "con": con
                }
        bs = test_pred
    del clf_if
    gc.collect()


print(best_params)
print(score)
print(classification_report(test_labels, bs, digits=4))

100%|██████████| 4/4 [00:12<00:00,  3.14s/it]


{'n_estimators': 10, 'con': 0.04}
0.598372828769923
              precision    recall  f1-score   support

           0     0.9668    0.9847    0.9757    199328
           1     0.3131    0.1708    0.2211      8130

    accuracy                         0.9528    207458
   macro avg     0.6400    0.5778    0.5984    207458
weighted avg     0.9412    0.9528    0.9461    207458



In [6]:
from pyod.models.pca import PCA
n_est = [5,10]
cont = [0.01, 0.04]
params = list(itertools.product(n_est, cont))
score = -1
bs = None

for n_est, con in tqdm.tqdm(params):
    clf_if = PCA(n_components=n_est, contamination=con)
    clf_if.fit(benign_train_samples)
    y_pred = clf_if.predict(test_samples)
    test_pred = y_pred

    f1 = f1_score(test_labels, test_pred, average='macro')

    if f1 > score:
        score = f1
        best_params = {'n_estimators': n_est,
                       "con": con
                }
        bs = test_pred
    del clf_if
    gc.collect()


print(best_params)
print(score)
print(classification_report(test_labels, bs, digits=4))

100%|██████████| 4/4 [00:13<00:00,  3.37s/it]


{'n_estimators': 5, 'con': 0.01}
0.5007065602509271
              precision    recall  f1-score   support

           0     0.9610    0.9870    0.9738    199328
           1     0.0546    0.0185    0.0276      8130

    accuracy                         0.9490    207458
   macro avg     0.5078    0.5027    0.5007    207458
weighted avg     0.9255    0.9490    0.9367    207458



In [7]:
n_est = [5,10]
cont = [0.01, 0.04]
params = list(itertools.product(n_est, cont))
score = -1
bs = None

for n_est, con in tqdm.tqdm(params):
    clf_if = PCA(n_components=n_est, contamination=con)
    clf_if.fit(normal_train_samples)
    y_pred = clf_if.predict(test_samples)
    test_pred = y_pred

    f1 = f1_score(test_labels, test_pred, average='macro')

    if f1 > score:
        score = f1
        best_params = {'n_estimators': n_est,
                       "con": con
                }
        bs = test_pred
    del clf_if
    gc.collect()


print(best_params)
print(score)
print(classification_report(test_labels, bs, digits=4))

100%|██████████| 4/4 [00:13<00:00,  3.45s/it]


{'n_estimators': 5, 'con': 0.01}
0.5009308096968799
              precision    recall  f1-score   support

           0     0.9610    0.9874    0.9741    199328
           1     0.0565    0.0185    0.0278      8130

    accuracy                         0.9495    207458
   macro avg     0.5087    0.5029    0.5009    207458
weighted avg     0.9256    0.9495    0.9370    207458



In [8]:
from sklearn.ensemble import IsolationForest
n_est = [20, 50]
cont = [0.01,0.04]
params = list(itertools.product(n_est, cont))
score = -1
bs = None

for n_est, con in tqdm.tqdm(params):
    clf_if = IsolationForest(n_estimators=n_est, contamination=con)
    clf_if.fit(benign_train_samples)
    y_pred = clf_if.predict(test_samples)
    test_pred = list(map(lambda x : 0 if x == 1 else 1, y_pred))

    f1 = f1_score(test_labels, test_pred, average='macro')

    if f1 > score:
        score = f1
        best_params = {'n_estimators': n_est,
                       "con": con
                }
        bs = test_pred
    del clf_if
    gc.collect()


print(best_params)
print(score)
print(classification_report(test_labels, bs, digits=4))

100%|██████████| 4/4 [00:24<00:00,  6.17s/it]


{'n_estimators': 20, 'con': 0.01}
0.6003777218970807
              precision    recall  f1-score   support

           0     0.9665    0.9891    0.9777    199328
           1     0.3735    0.1590    0.2231      8130

    accuracy                         0.9566    207458
   macro avg     0.6700    0.5741    0.6004    207458
weighted avg     0.9432    0.9566    0.9481    207458



In [9]:
from sklearn.ensemble import IsolationForest

n_est = [20, 50]
cont = [ 0.01,0.04]
params = list(itertools.product(n_est, cont))
score = -1
bs = None

for n_est, con in tqdm.tqdm(params):
    clf_if = IsolationForest(n_estimators=n_est, contamination=con)
    clf_if.fit(normal_train_samples)
    y_pred = clf_if.predict(test_samples)
    test_pred = list(map(lambda x : 0 if x == 1 else 1, y_pred))

    f1 = f1_score(test_labels, test_pred, average='macro')

    if f1 > score:
        score = f1
        best_params = {'n_estimators': n_est,
                       "con": con
                }
        bs = test_pred
    del clf_if
    gc.collect()


print(best_params)
print(score)
print(classification_report(test_labels, bs, digits=4))

100%|██████████| 4/4 [00:25<00:00,  6.43s/it]


{'n_estimators': 20, 'con': 0.04}
0.554540063557621
              precision    recall  f1-score   support

           0     0.9653    0.9622    0.9637    199328
           1     0.1401    0.1509    0.1453      8130

    accuracy                         0.9304    207458
   macro avg     0.5527    0.5566    0.5545    207458
weighted avg     0.9329    0.9304    0.9317    207458

